# LS PremierLink LE125 — 굽힘손실 FEM 계산 및 사양 비교

2026-11-03 · Velamuri (2018) 기반 2D scalar P1 FEM + complex PML · **실제 LE125 RIP/실험값으로 검증되지 않은 가상 Step-index 모델**입니다.

**제조사 제공 표 시험조건:** **굽힘 반경 R=30 mm**, **100회 권선**, 1550/1625 nm 각각 **≤0.1 dB**. 솔버 `bend_diameter_mm`는 **직경**이므로 반드시 `60`을 입력합니다.

**근사 모델:** MFD≈12.5 µm에 맞춰 설정된 `a=5.5 µm`, `Δn=0.0034785793846155`. 실측 LE125 굴절률 분포(RIP)가 아닙니다. Full-vector EWFD, GEBT 응력장, 실제 클래딩·코팅 계면은 미구현입니다.

사전에 실행한 시연 결과: h=0.55 µm/PML=7λ일 때 1550 nm **0.974088 dB/100 turns**, 1625 nm **4.979979 dB/100 turns**. 단, **수렴 기준 미충족**, 제품 적합성 판단 불가.

[전체 비교보고서](./LE125_BENDING_VALIDATION_2026-11-03.md) · [1550 nm JSON](./LE125_BEND_1550_UNVALIDATED.json) · [1625 nm JSON](./LE125_BEND_1625_UNVALIDATED.json)

## 1. 웹 모드해석과 동일 저장소의 2D FEM 엔진 고정

아래 셀에는 GitHub `COMSOL/universal_silica_v01/bending_fem/solver.py` 코드를 **소스 Blob SHA `2a6dcccd0f12754e4a6361b05cd0e303beab3366`** 기준으로 내장했습니다. 외부 사이트 접속 없이 같은 코드를 재실행할 수 있습니다. Python NumPy/SciPy 환경이 필요합니다.

**주의:** 이 2D scalar FEM은 기존 웹의 1D LP01 솔버와 별도입니다.

In [ ]:
from pathlib import Path
from dataclasses import replace
import json, hashlib, numpy as np, pandas as pd, matplotlib.pyplot as plt
source="\"\"\"2D scalar P1 triangular FEM with complex PML for optical-fiber bend modes.\n\nResearch prototype inspired by Velamuri et al. (COMSOL Conference 2018).\nThis is a real 2-D eigenmode numerical solve, NOT their full-vector EWFD/GEBT,\nnor a verified model of any commercial cable or actual physical coating.\nThe PML is on a shortened glass-only computational window: do NOT interpret\nits leakage rate as certified macrobend loss without mesh/PML/window sweeps.\n\nEquation: div(C grad E) + k0^2 n_eq^2 s_x s_y E = beta^2 s_x s_y E\nC=diag(s_y/s_x, s_x/s_y), n_eq(x,y)=n_stress(r)(1+x/R).\nOuter boundary is zero after the complex-coordinate-stretched PML.\n\"\"\"\nfrom __future__ import annotations\nfrom dataclasses import dataclass, asdict, replace\nfrom math import pi, log\nimport numpy as np\nfrom scipy.sparse import coo_matrix\nfrom scipy.sparse.linalg import eigs, splu, LinearOperator\n\n_B = (.6961663,.4079426,.8974794)\n_C = (.0684043**2,.1162414**2,9.896161**2)\n\ndef silica_index(wavelength_nm):\n    w = wavelength_nm / 1000.\n    if not (.21 < w < 3.7):\n        raise ValueError(\"silica Sellmeier wavelength requires 210-3700 nm\")\n    return np.sqrt(1 + sum(b*w*w/(w*w-c) for b,c in zip(_B,_C)))\n\n@dataclass(frozen=True)\nclass Fiber:\n    wavelength_nm:float = 1550.\n    core_radius_um:float = 4.1\n    delta_n_core:float = .005\n    profile:str = \"step\"\n    trench_start_um:float = 7.\n    trench_end_um:float = 12.\n    trench_delta_n:float = -.002\n    dip_radius_um:float = 0.\n    dip_delta_n:float = 0.\n    bend_diameter_mm:float = 10.\n    stress_mode:str = \"none\"\n    poisson_ratio:float = .17\n    p11:float = .113\n    p12:float = .252\n    window_um:float = 22.\n    pml_thickness_um:float = 10.85\n    pml_strength:float = 2.\n    pml_exponent:int = 3\n    grid_h_um:float = .7\n    n_modes:int = 10\n    max_points:int = 220\n\ndef validate(f):\n    if f.profile not in (\"step\",\"trench\"):\n        raise ValueError(\"profile must be step or trench\")\n    if f.stress_mode not in (\"none\",\"beam_proxy\"):\n        raise ValueError(\"stress_mode must be none or beam_proxy (NOT GEBT)\")\n    for key in (\"wavelength_nm\",\"core_radius_um\",\"delta_n_core\",\"bend_diameter_mm\",\n                \"window_um\",\"pml_thickness_um\",\"pml_strength\",\"grid_h_um\"):\n        if not np.isfinite(getattr(f,key)):\n            raise ValueError(\"invalid \"+key)\n    if f.core_radius_um <= 0 or f.delta_n_core <= 0 or f.window_um <= 2*f.core_radius_um:\n        raise ValueError(\"need positive core/index step and window >2*core radius\")\n    if not 0 <= f.bend_diameter_mm <= 1000:\n        raise ValueError(\"bend diameter must be 0..1000 mm, zero is straight\")\n    if min(f.pml_thickness_um,f.pml_strength,f.grid_h_um) <= 0:\n        raise ValueError(\"PML thickness/strength and mesh size must be positive\")\n    if not 2 <= f.n_modes <= 15:\n        raise ValueError(\"n_modes must be 2..15\")\n    if f.stress_mode == \"beam_proxy\" and not 0 <= f.poisson_ratio < .5:\n        raise ValueError(\"invalid Poisson ratio\")\n    if f.profile == \"trench\" and not (\n        f.core_radius_um < f.trench_start_um < f.trench_end_um < f.window_um\n    ):\n        raise ValueError(\"trench must lie inside the non-PML computational window\")\n    if f.bend_diameter_mm and (\n        f.window_um + f.pml_thickness_um >= 500*f.bend_diameter_mm\n    ):\n        raise ValueError(\"2D computational half-width must be smaller than bend radius\")\n    silica_index(f.wavelength_nm)\n    return f\n\ndef index_field(f,x,y):\n    r=np.hypot(x,y)\n    n=silica_index(f.wavelength_nm) + f.delta_n_core*(r < f.core_radius_um)\n    if f.profile==\"trench\":\n        n=n+f.trench_delta_n*((r>=f.trench_start_um)&(r<f.trench_end_um))\n    if f.dip_radius_um>0:\n        n=n+f.dip_delta_n*(r<f.dip_radius_um)\n    if f.bend_diameter_mm>0:\n        eps=x/(f.bend_diameter_mm*500.)\n        if f.stress_mode==\"beam_proxy\":\n            n=n*(1-.5*n*n*(f.p11*eps-2*f.p12*f.poisson_ratio*eps))\n        n=n*(1+eps)\n    return n\n\ndef stretch(f,x):\n    frac=np.maximum((np.abs(x)-f.window_um)/f.pml_thickness_um,0)\n    return 1+1j*f.pml_strength*frac**f.pml_exponent\n\ndef assemble(f):\n    validate(f)\n    half=f.window_um+f.pml_thickness_um\n    nx=max(11,int(np.ceil(2*half/f.grid_h_um))+1)\n    if nx%2==0: nx+=1\n    if nx>f.max_points:\n        raise ValueError(\"mesh too large; increase grid_h_um or max_points\")\n    xx=np.linspace(-half,half,nx)\n    h=float(xx[1]-xx[0])\n    i,j=np.meshgrid(np.arange(nx-1),np.arange(nx-1),indexing=\"ij\")\n    v=(i*nx+j).ravel()\n    triangles=np.vstack([\n        np.stack([v,v+nx,v+nx+1],axis=1),\n        np.stack([v,v+nx+1,v+1],axis=1)])\n    nt=len(triangles)\n    X,Y=np.meshgrid(xx,xx,indexing=\"ij\")\n    coords=np.stack([X.ravel(),Y.ravel()],axis=1)\n    pts=coords[triangles]\n    x,y=pts[:,:,0],pts[:,:,1]\n    cross=((x[:,1]-x[:,0])*(y[:,2]-y[:,0])-\n           (x[:,2]-x[:,0])*(y[:,1]-y[:,0]))\n    area=np.abs(cross)/2\n    gx=np.stack([y[:,1]-y[:,2],y[:,2]-y[:,0],y[:,0]-y[:,1]],axis=1)/cross[:,None]\n    gy=np.stack([x[:,2]-x[:,1],x[:,0]-x[:,2],x[:,1]-x[:,0]],axis=1)/cross[:,None]\n    xc,yc=x.mean(axis=1),y.mean(axis=1)\n    sx,sy=stretch(f,xc),stretch(f,yc)\n    b=sx*sy\n    k0=2*pi/(f.wavelength_nm/1000)\n    n=index_field(f,xc,yc)\n    M=(area*b)[:,None,None]*(np.ones((3,3))+np.eye(3))[None,:,:]/12\n    A=(k0*k0*n*n)[:,None,None]*M-area[:,None,None]*(\n        (sy/sx)[:,None,None]*gx[:,:,None]*gx[:,None,:]+\n        (sx/sy)[:,None,None]*gy[:,:,None]*gy[:,None,:])\n    nodes=nx*nx\n    rows=np.broadcast_to(triangles[:,:,None],(nt,3,3)).ravel()\n    cols=np.broadcast_to(triangles[:,None,:],(nt,3,3)).ravel()\n    matA=coo_matrix((A.ravel(),(rows,cols)),shape=(nodes,nodes)).tocsr()\n    matM=coo_matrix((M.ravel(),(rows,cols)),shape=(nodes,nodes)).tocsr()\n    idx=np.arange(nx)[:,None]\n    jdx=np.arange(nx)[None,:]\n    interior=(idx>0)&(idx<nx-1)&(jdx>0)&(jdx<nx-1)\n    keep=np.flatnonzero(interior.ravel())\n    return matA[keep,:][:,keep],matM[keep,:][:,keep],coords[keep],nx,h\n\ndef solve(f, include_field=False):\n    A,M,coords,nx,h=assemble(f)\n    k0=2*pi/(f.wavelength_nm/1000)\n    n_guess=silica_index(f.wavelength_nm)+.45*f.delta_n_core\n    sigma=(k0*n_guess)**2\n    # For complex PML mass, SciPy eigs(A,M=M,sigma) is not safe: M is\n    # non-Hermitian/non-SPD. Apply generalized shift-invert explicitly.\n    lu=splu((A-sigma*M).tocsc())\n    operator=LinearOperator(A.shape,dtype=complex,matvec=lambda v:lu.solve(M@v))\n    mu,vecs=eigs(operator,k=f.n_modes,which=\"LM\",tol=1e-9,maxiter=1500)\n    beta_sq=sigma+1/mu\n    neffs=np.sqrt(beta_sq)/k0\n    r=np.hypot(coords[:,0],coords[:,1])\n    actual=(np.abs(coords).max(axis=1)<f.window_um)\n    nearcore=(r<2*f.core_radius_um)\n    modes=[]\n    for idx,neff in enumerate(neffs):\n        power=np.abs(vecs[:,idx])**2\n        core_fraction=float(power[nearcore].sum()/max(power[actual].sum(),1e-30))\n        physical_fraction=float(power[actual].sum()/max(power.sum(),1e-30))\n        modes.append(dict(index=idx,neff_real=float(neff.real),\n            neff_imag=float(neff.imag),core_fraction=core_fraction,\n            physical_fraction=physical_fraction))\n    selected=max(modes,key=lambda mode:mode[\"core_fraction\"])\n    imag=selected[\"neff_imag\"]\n    alpha_db_m=20/log(10)*k0*1e6*imag\n    bend_db_turn=(alpha_db_m*pi*f.bend_diameter_mm/1000\n                  if f.bend_diameter_mm>0 and imag>=0 else None)\n    result=dict(\n        input=asdict(f),neff_real=selected[\"neff_real\"],neff_imag=imag,\n        bend_loss_dB_per_turn=bend_db_turn,loss_dB_per_m=alpha_db_m,\n        core_fraction=selected[\"core_fraction\"],\n        physical_fraction=selected[\"physical_fraction\"],\n        grid_n=nx,h_um=h,unknowns=A.shape[0],ranked_modes=modes,\n        nonnegative_imag=bool(imag>=0),is_fully_validated=False,\n        solver_level=\"2D scalar P1 triangular FEM with complex PML (NOT full-vector COMSOL/GEBT)\",\n        limitations=[\n            \"Square PML truncates actual 125um cladding; coating not modeled\",\n            \"Requires independent PML thickness, mesh, domain and mode-tracking convergence\",\n            \"Optional linear beam/Poisson stress proxy is NOT GEBT\",\n            \"No experimental product or paper profile-level validation\"])\n    if include_field:\n        chosen=selected[\"index\"]\n        # Resize only the DISPLAY intensity; the mode was computed on the full mesh.\n        intensity=np.abs(vecs[:,chosen].reshape((nx-2,nx-2)))**2\n        intensity=intensity/max(float(intensity.max()),1e-30)\n        stride=max(1,int(np.ceil((nx-2)/101)))\n        result[\"mode_intensity_image\"]=intensity[::stride,::stride].tolist()\n        result[\"mode_image_stride\"]=stride\n        result[\"mode_image_origin_um\"]=-f.window_um-f.pml_thickness_um+h\n        result[\"mode_image_h_um\"]=h*stride\n    return result\n\ndef convergence_study(f,mesh_values=(.8,.65,.55),\n                      pml_values=(6.2,9.3,10.85),tolerance_percent=2.):\n    \"\"\"Real independent FEM reruns; not an assertion of convergence.\"\"\"\n    base=solve(f)\n    mesh=[dict(grid_h_requested_um=h,**solve(replace(f,grid_h_um=h)))\n          for h in mesh_values]\n    pml=[dict(pml_thickness_requested_um=t,\n              **solve(replace(f,pml_thickness_um=t))) for t in pml_values]\n    def compare(a,b):\n        x=a[\"bend_loss_dB_per_turn\"];y=b[\"bend_loss_dB_per_turn\"]\n        if x is None or y is None or y<=0:return None\n        return 100*abs(x-y)/abs(y)\n    mesh_change=compare(mesh[-2],mesh[-1]) if len(mesh)>=2 else None\n    pml_change=compare(pml[-2],pml[-1]) if len(pml)>=2 else None\n    passed=(mesh_change is not None and pml_change is not None\n        and mesh_change<=tolerance_percent and pml_change<=tolerance_percent)\n    return dict(base=base,mesh=mesh,pml=pml,\n                mesh_last_two_change_pct=mesh_change,\n                pml_last_two_change_pct=pml_change,\n                tolerance_percent=tolerance_percent,\n                last_two_sweeps_passed=passed,\n                is_fully_validated=False,\n                caveat=\"Even a pass requires domain-size, polarization, mode tracking, cladding and experimental convergence.\")\n\nif __name__==\"__main__\":\n    import argparse,json,time\n    parser=argparse.ArgumentParser(description=__doc__)\n    for name,field in Fiber.__dataclass_fields__.items():\n        val=field.default\n        parser.add_argument(\"--\"+name.replace(\"_\",\"-\"),type=type(val),default=val)\n    parser.add_argument(\"--convergence\",action=\"store_true\",\n                        help=\"repeat independent runs on 3 grids and 3 PML thicknesses\")\n    args=vars(parser.parse_args())\n    isstudy=args.pop(\"convergence\")\n    f=Fiber(**args)\n    start=__import__(\"time\").time()\n    result=convergence_study(f) if isstudy else solve(f)\n    result[\"elapsed_s\"]=__import__(\"time\").time()-start\n    print(json.dumps(result,indent=2))\n"
raw=source.encode('utf-8')
git_blob_sha=hashlib.sha1(b'blob '+str(len(raw)).encode()+b'\\0'+raw).hexdigest()
print('Embedded FEM source SHA:',git_blob_sha)
print('Expected source SHA: 2a6dcccd0f12754e4a6361b05cd0e303beab3366')
assert git_blob_sha=='2a6dcccd0f12754e4a6361b05cd0e303beab3366', 'Embedded source differs from committed solver!'
Path('le125_bend2d_solver.py').write_text(source, encoding='utf-8')
from le125_bend2d_solver import Fiber,solve
try:
 from threadpoolctl import threadpool_limits
 threadpool_limits(limits=1)
except ImportError:
 pass
print('2D scalar P1 FEM/PML source ready.')

## 2. LS 사양과 동일한 반경·파장·권선 조건으로 독립 재계산

각 파장에서 **메시 0.90, 0.70, 0.55 µm**의 7λ PML과 **0.55 µm 메시 / 6λ PML**을 별도 eigenmode solve로 계산합니다. **물리적 클래딩·코팅/실제 RIP 미반영**이라는 한계를 지닌 코드입니다.

가장 미세한 메시의 손실을 최종적인 제품 예측값으로 해석하지 마세요.

In [ ]:
wavelengths=[1550,1625]
all_runs={}
for nm in wavelengths:
 print(f'--- Running {nm} nm ---')
 rows=[]
 for h in (0.90,0.70,0.55):
  for pml_mult in ((6,7) if h==0.55 else (7,)):
   m=Fiber(wavelength_nm=nm,core_radius_um=5.5,
           delta_n_core=0.0034785793846155,profile='step',
           bend_diameter_mm=60,window_um=63,
           pml_thickness_um=pml_mult*nm/1000,
           grid_h_um=h,max_points=320,n_modes=10,
           stress_mode='none')
   out=solve(m)
   row=dict(wavelength_nm=nm,h_requested_um=h,
            pml_lambda=pml_mult,output=out)
   rows.append(row)
   print('h',h,'PML',pml_mult,'loss(100 turns)=',
         None if out['bend_loss_dB_per_turn'] is None else 100*out['bend_loss_dB_per_turn'])
 all_runs[nm]=rows
print('Runs completed. All losses remain hypothetical and unqualified.')

In [ ]:
summary=[]
for nm,rows in all_runs.items():
 def get(h,p):
  return next(r['output'] for r in rows if abs(r['h_requested_um']-h)<1e-8 and r['pml_lambda']==p)
 fine=get(0.55,7);prev=get(0.7,7);pml6=get(0.55,6)
 val=100*fine['bend_loss_dB_per_turn']
 mesh_pct=100*abs(val-100*prev['bend_loss_dB_per_turn'])/val
 pml_pct=100*abs(val-100*pml6['bend_loss_dB_per_turn'])/val
 valid=mesh_pct<=2 and pml_pct<=2
 summary.append({'wavelength_nm':nm,'nominal_limit_db_per_100turns':0.1,
  'uncalibrated_model_loss_db_per_100turns':val,
  'Im_neff':fine['neff_imag'],'mesh_last_two_change_pct':mesh_pct,
  'pml_last_two_change_pct':pml_pct,'screening_passed':valid,
  'product_pass_fail':'NOT DETERMINABLE'})
summary_df=pd.DataFrame(summary)
display(summary_df)
if not summary_df['screening_passed'].all():
 print('WARNING: Mesh/PML convergence criterion NOT met. Product pass/fail CANNOT be determined.')

In [ ]:
fig,ax=plt.subplots(figsize=(10,5))
for nm,rows in all_runs.items():
 mesh=[r for r in rows if r['pml_lambda']==7]
 ax.plot([r['h_requested_um'] for r in mesh],
         [100*r['output']['bend_loss_dB_per_turn'] for r in mesh],
         'o-',label=f'{nm} nm (unvalidated surrogate FEM)')
ax.axhline(0.1,color='gray',linestyle='--',label='LE125 datasheet upper limit 0.1 dB / 100 turns')
ax.set(xlabel='Mesh size h (µm)',ylabel='Bend loss (dB / 100 turns)',
 title='LE125 R30 mm ×100 turns — UNVALIDATED step-index scalar FEM')
ax.legend();ax.grid(alpha=0.3);ax.invert_xaxis()
plt.tight_layout();plt.show()

In [ ]:
# Export browser-importable records for both wavelengths; include a FALSE convergence flag.
for nm,rows in all_runs.items():
 def get(h,p):
  return next(r['output'] for r in rows if abs(r['h_requested_um']-h)<1e-8 and r['pml_lambda']==p)
 mesh=[{'grid_h_requested_um':h,**get(h,7)} for h in (0.9,0.7,0.55)]
 pml=[{'pml_thickness_requested_um':p*nm/1000,**get(0.55,p)} for p in (6,7)]
 fine=mesh[-1]['bend_loss_dB_per_turn']
 mesh_pct=100*abs(fine-mesh[-2]['bend_loss_dB_per_turn'])/fine
 pml_pct=100*abs(pml[-1]['bend_loss_dB_per_turn']-pml[-2]['bend_loss_dB_per_turn'])/fine
 payload={'file_kind':'LSCNS_BEND_2D_SCALAR_FEM_PML_V1',
 'source_git_blob_sha':git_blob_sha,
 'product':'LS PremierLink LE125 (surrogate model ONLY)',
 'base':mesh[-1],'mesh':mesh,'pml':pml,
 'mesh_last_two_change_pct':mesh_pct,'pml_last_two_change_pct':pml_pct,
 'last_two_sweeps_passed':mesh_pct<=2 and pml_pct<=2,
 'tolerance_percent':2,'is_fully_validated':False,
 'cannot_assess_actual_product_compliance':True,
 'warning':'Real LE125 index RIP and physical coating are unknown; do not use as actual product bending loss'}
 filename=f'LE125_BEND_{nm}_UNVALIDATED_recomputed.json'
 Path(filename).write_text(json.dumps(payload,indent=2),encoding='utf-8')
 print(filename, 'saved, product qualified?',False)
summary_df.to_csv('LE125_BENDING_COMPARISON_UNVALIDATED.csv',index=False)
print('CSV saved. This is not manufacturer certification.')

## 3. 기술적 결론

이번 검증은 제조사 시험조건과 **파장·반경·권선 수만** 맞춘 것입니다. **실제 LE125 RIP, 실물 코팅 계면, Full-vector Maxwell, GEBT 응력장, PML 수렴 및 실측 결과**는 일치시키지 못했습니다.

기준 `0.1 dB`를 단순히 벗어난 잠정 FEM 숫자를 실제 제품의 불합격으로 선언하면 안 됩니다. 원인 분석의 최우선은 **실제 코어/트렌치 굴절률 프로파일 확보**와 **물리 외부 경계/PML 수렴**입니다.